# Smart Desktop Keyboard — IndicTrans2 Model Conversion (FIXED)

Exports IndicTrans2 EN→HI (`indictrans2-en-indic-dist-200M`) to ONNX INT8 for fully offline CPU use.

**Runtime → Change runtime type → CPU is fine.**

Run all cells **top to bottom in a single session**. Download the zip at the end.

---
### What each cell does
| Cell | Purpose |
|------|---------|
| 0 | Shared constants — SRC/REF sentences, language codes, paths |
| 1 | Install all dependencies |
| 2 | Load PyTorch model, verify language-tagged tokenization, record baseline BLEU + chrF++ |
| 3 | Export to ONNX (three graphs: encoder / decoder / decoder_with_past) + smoke-test graphs |
| 4 | INT8 dynamic quantization → standard filenames, verify MatMulInteger nodes applied |
| 5 | Validate quantized model quality (BLEU + chrF++) + latency benchmark + zip for download |

---
### Fixes applied vs original notebook
1. **Shared constants cell (Cell 0)** — SRC/REF/paths defined once; no NameError if cells re-run out of order.
2. **Language tags** — `src_lang`/`forced_bos_token_id` set correctly on every tokenizer call.
3. **ONNX filenames kept standard** — quantized files keep original names so `ORTModelForSeq2SeqLM.from_pretrained()` finds them without extra arguments.
4. **`optimize_model=False`** — avoids fusing nodes before quantization, which silently skips INT8 on fused ops.
5. **MatMulInteger verification** — confirms quantization actually applied (non-zero count).
6. **Tokenizer loaded from QUANT_DIR** — guarantees same custom code used at export is used at inference.
7. **`trust_remote_code=True`** added to `ORTModelForSeq2SeqLM.from_pretrained()`.
8. **Required tokenizer files verified** after copy.
9. **ONNX graph smoke-test** after export via `onnx.checker`.
10. **`num_beams=1` (greedy)** for ONNX inference — beam search on ORT runs as a Python loop (4× decoder calls per step); greedy is the correct production choice for a keyboard app.
11. **`max_new_tokens=64`** instead of `max_length=256` — `max_length` counts input+output tokens together; `max_new_tokens` is the correct cap for output length.
12. **`repetition_penalty=1.3`** — prevents repeated-token degeneration.
13. **chrF++** added alongside BLEU — more robust metric for morphologically rich Hindi.
14. **Per-sentence BLEU drop check** — catches catastrophic failures on individual sentences that aggregate BLEU masks.
15. **P50/P95 latency benchmark** — per-sentence, greedy, reflects real keyboard-app conditions.
16. **Thread tuning** — `intra_op_num_threads=2` to avoid background CPU saturation in a desktop app.
17. **`repetition_penalty`** passed to baseline `generate()` for a fair like-for-like comparison.


In [2]:
# ── CELL 0 — Shared constants (run this first; all other cells depend on it) ──────────────
#
# Defining SRC/REF/paths here means any cell can be re-run in isolation
# without a NameError — fixes Bug 1 from the original notebook.

import os

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = 'ai4bharat/indictrans2-en-indic-dist-200M'

# IndicTrans2 language codes (Flores-200 format used by this tokenizer)
SRC_LANG = 'eng_Latn'   # English
TGT_LANG = 'hin_Deva'   # Hindi (Devanagari)

# ── Paths ─────────────────────────────────────────────────────────────────────
ONNX_DIR  = '/content/indictrans2_onnx'
QUANT_DIR = '/content/indictrans2_onnx_int8'
ZIP_PATH  = '/content/indictrans2_int8'

os.makedirs(ONNX_DIR,  exist_ok=True)
os.makedirs(QUANT_DIR, exist_ok=True)

# ── Test sentences ────────────────────────────────────────────────────────────
# Short phrases representative of keyboard / messaging use-cases.
# These are used both for baseline PyTorch scoring AND quantized ORT scoring
# so the comparison is always apples-to-apples.
SRC = [
    'Hello, how are you?',
    'I will be late today.',
    'Please send me the document.',
    'The meeting is at 3 PM.',
    'I love you.',
    'Thank you so much.',
    'Can you help me?',
    'I am feeling sick today.',
    'The weather is very nice.',
    'Let us go for a walk.',
    'I will call you tomorrow.',
    'Happy birthday!',
    'Please wait for me.',
    'I am very tired.',
    'What is your name?',
    'I need to talk to you.',
    'The food is delicious.',
    'See you tomorrow.',
    'I am going to the office.',
    'Please take care of yourself.',
]

# Reference translations (used for BLEU / chrF++ regression, not absolute scoring).
# NOTE: BLEU against a single reference is a regression signal only.
# A lower BLEU vs these refs does NOT mean the model output is wrong —
# valid paraphrases are penalised. Use chrF++ for a more reliable signal
# and always do a manual spot-check with a native speaker before shipping.
REF = [
    'नमस्ते, आप कैसे हैं?',
    'मैं आज देर से आऊँगा।',
    'कृपया मुझे दस्तावेज़ भेजें।',
    'बैठक 3 बजे है।',
    'मैं तुमसे प्यार करता हूँ।',
    'बहुत बहुत धन्यवाद।',
    'क्या आप मेरी मदद कर सकते हैं?',
    'मैं आज बीमार महसूस कर रहा हूँ।',
    'मौसम बहुत अच्छा है।',
    'चलो टहलने चलते हैं।',
    'मैं कल आपको फोन करूँगा।',
    'जन्मदिन मुबारक हो!',
    'कृपया मेरा इंतजार करें।',
    'मैं बहुत थका हुआ हूँ।',
    'आपका नाम क्या है?',
    'मुझे आपसे बात करनी है।',
    'खाना बहुत स्वादिष्ट है।',
    'कल मिलते हैं।',
    'मैं ऑफिस जा रहा हूँ।',
    'कृपया अपना ख्याल रखें।',
]

assert len(SRC) == len(REF), 'SRC and REF must be the same length'
print(f'Cell 0 done. {len(SRC)} sentence pairs loaded.')
print(f'ONNX_DIR  : {ONNX_DIR}')
print(f'QUANT_DIR : {QUANT_DIR}')

Cell 0 done. 20 sentence pairs loaded.
ONNX_DIR  : /content/indictrans2_onnx
QUANT_DIR : /content/indictrans2_onnx_int8


In [3]:
# ── Authenticate with HuggingFace (required for gated models) ────────────────
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [5]:
!pip install -q \
    transformers==4.41.0 \
    onnxruntime==1.20.1 \
    onnx==1.17.0 \
    sacrebleu \
    sentencepiece
# No numpy pin — let Colab's numpy 2.x stay in place
# No optimum — we export manually in Cell 3

!pip uninstall -y diffusers optimum 2>/dev/null; echo "done"

print("Restart runtime, then re-run Cell 0 and continue from Cell 2.")

done
Restart runtime, then re-run Cell 0 and continue from Cell 2.


In [6]:
# ── CELL 2 — Load PyTorch model and record baseline metrics ───────────────────
#
# FIX: Language tags are now set correctly on every tokenizer call.
# IndicTrans2 uses Flores-200 language codes and requires:
#   tokenizer.src_lang = SRC_LANG      → tells the encoder which language to expect
#   forced_bos_token_id                → forces the decoder to start in the target language
# Without these the model may output the wrong language or garbage tokens.
#
# FIX: repetition_penalty=1.3 added to prevent repeated-token degeneration.
# FIX: max_new_tokens used instead of max_length.
#      max_length=256 counts input+output together, so a 100-token input
#      leaves only 156 tokens for output. max_new_tokens=128 caps only output.
#
# NOTE: baseline still uses num_beams=4 for PyTorch because PyTorch beam search
# is fused and fast. The ORT model will use greedy (num_beams=1) — see Cell 5.
# We record both greedy and beam baseline here so the comparison is fair.

# ── CELL 2 — Load PyTorch model and record baseline metrics ───────────────────

import torch
import sacrebleu
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

assert 'SRC' in dir(), 'Run Cell 0 first.'

print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()
print('Model loaded.')

# ── FIX: IndicTransTokenizer uses convert_tokens_to_ids(), not lang_code_to_id ──
# Verify the token exists before generating so the error is clear if it doesn't.
# forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
# assert forced_bos_token_id != tokenizer.unk_token_id, (
#     f"'{TGT_LANG}' was not found in the tokenizer vocabulary. "
#     f"Check TGT_LANG in Cell 0 — must be a Flores-200 code like 'hin_Deva'."
# )
# print(f'forced_bos_token_id for {TGT_LANG}: {forced_bos_token_id}')

# ── Helper: language-aware translation ───────────────────────────────────────
def translate_pt(sentences, mdl, tok, num_beams=1):
    tagged = [f"{SRC_LANG} {TGT_LANG} {s}" for s in sentences]

    inp = tok(
        tagged,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=256,
    )
    with torch.no_grad():
        out = mdl.generate(
            **inp,
            # forced_bos_token_id removed — IndicTrans2 routes via input prefix tags,
            # not via a forced decoder token. Forcing id=15 was injecting 'छे' as
            # the first output token on every sentence.
            num_beams=num_beams,
            max_new_tokens=128,
            repetition_penalty=1.3,
        )
    return tok.batch_decode(out, skip_special_tokens=True)

# ── Baseline: beam=4 ──────────────────────────────────────────────────────────
print('\nRunning baseline (beam=4) ...')
hyps_beam = translate_pt(SRC, model, tokenizer, num_beams=4)

baseline_bleu = sacrebleu.corpus_bleu(hyps_beam, [REF]).score
baseline_chrf = sacrebleu.corpus_chrf(hyps_beam, [REF]).score
print(f'Baseline BLEU  (beam=4): {baseline_bleu:.2f}')
print(f'Baseline chrF++ (beam=4): {baseline_chrf:.2f}')

# ── Baseline: greedy ──────────────────────────────────────────────────────────
print('\nRunning baseline (greedy / beam=1) ...')
hyps_greedy = translate_pt(SRC, model, tokenizer, num_beams=1)

baseline_bleu_greedy = sacrebleu.corpus_bleu(hyps_greedy, [REF]).score
baseline_chrf_greedy = sacrebleu.corpus_chrf(hyps_greedy, [REF]).score
print(f'Baseline BLEU  (greedy): {baseline_bleu_greedy:.2f}')
print(f'Baseline chrF++ (greedy): {baseline_chrf_greedy:.2f}')

# ── Spot-check ────────────────────────────────────────────────────────────────
print('\nSpot-check (beam=4):')
for s, h, r in zip(SRC[:3], hyps_beam[:3], REF[:3]):
    print(f'  EN  : {s}')
    print(f'  HI  : {h}')
    print(f'  REF : {r}')
    print()

Loading ai4bharat/indictrans2-en-indic-dist-200M ...
Model loaded.

Running baseline (beam=4) ...
Baseline BLEU  (beam=4): 41.23
Baseline chrF++ (beam=4): 78.45

Running baseline (greedy / beam=1) ...
Baseline BLEU  (greedy): 37.64
Baseline chrF++ (greedy): 82.52

Spot-check (beam=4):
  EN  : Hello, how are you?
  HI  : नमस्ते, आप कैसे हैं?
  REF : नमस्ते, आप कैसे हैं?

  EN  : I will be late today.
  HI  : मुझे आज देर हो जाएगी ।
  REF : मैं आज देर से आऊँगा।

  EN  : Please send me the document.
  HI  : कृपया मुझे दस्तावेज़ भेजें ।
  REF : कृपया मुझे दस्तावेज़ भेजें।



In [7]:
!pip install -U -q huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.19.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.9.0 which is incompatible.
transformers 4.41.0 requires huggingface-hub<1.0,>=0.23.0, but you have huggingface-hub 1.9.0 which is incompatible.


In [8]:
# ── CELL 3 — Export encoder + decoder to ONNX via torch.onnx ─────────────────
#
# IndicTrans is a custom architecture not supported by optimum-cli.
# We export the three subgraphs manually:
#   encoder_model.onnx            — encodes source tokens
#   decoder_model.onnx            — first decode step (no KV cache)
#   decoder_with_past_model.onnx  — subsequent steps (with KV cache)
#
# NOTE: decoder_with_past_model is complex to export due to dynamic KV-cache
# shapes. For a keyboard/on-device use case, a single fused decoder export
# (no KV-cache split) is simpler and only ~10% slower on short sentences.
# We export that here as decoder_model.onnx and skip decoder_with_past_model.
import torch
import onnx
import os
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

assert 'SRC' in dir(), 'Run Cell 0 first.'
assert 'model' in dir(), 'Run Cell 2 first (need loaded PyTorch model).'

os.makedirs(ONNX_DIR, exist_ok=True)

# ── 1. Export encoder ─────────────────────────────────────────────────────────
print('Exporting encoder ...')

tagged   = [f"{SRC_LANG} {TGT_LANG} Hello, how are you?"]
enc_inp  = tokenizer(tagged, return_tensors='pt', padding=True, truncation=True, max_length=256)
input_ids      = enc_inp['input_ids']
attention_mask = enc_inp['attention_mask']

class EncoderWrapper(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.enc = m.model.encoder
    def forward(self, input_ids, attention_mask):
        return self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

enc_wrapper = EncoderWrapper(model).eval()

torch.onnx.export(
    enc_wrapper,
    (input_ids, attention_mask),
    f'{ONNX_DIR}/encoder_model.onnx',
    input_names  = ['input_ids', 'attention_mask'],
    output_names = ['last_hidden_state'],
    dynamic_axes = {
        'input_ids':        {0: 'batch', 1: 'seq'},
        'attention_mask':   {0: 'batch', 1: 'seq'},
        'last_hidden_state':{0: 'batch', 1: 'seq'},
    },
    opset_version   = 14,
    do_constant_folding = True,
    dynamo = False,
)
print('  encoder_model.onnx ✓')

# ── 2. Export decoder (fused, no KV-cache split) ──────────────────────────────
print('Exporting decoder ...')

with torch.no_grad():
    encoder_hidden = enc_wrapper(input_ids, attention_mask)

# decoder input: just the pad/bos token to start
decoder_input_ids = torch.tensor([[model.config.decoder_start_token_id or 1]])

# ── Re-export decoder with correct dynamic seq dim ───────────────────────────
# The original export used a single-token input, so the graph only knows
# how to process length-1 sequences. We need to re-export with a multi-token
# example so the graph generalises to growing sequences.

import torch, onnx, os

class DecoderWrapper(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.model = m
    def forward(self, decoder_input_ids, encoder_hidden_states, encoder_attention_mask):
        out = self.model.model.decoder(
            input_ids             = decoder_input_ids,
            encoder_hidden_states = encoder_hidden_states,
            encoder_attention_mask= encoder_attention_mask,
        )
        return self.model.lm_head(out.last_hidden_state)

dec_wrapper = DecoderWrapper(model).eval()

# Export with a MULTI-TOKEN example (length 4) so the tracer sees a real seq dim
tagged   = [f"{SRC_LANG} {TGT_LANG} Hello, how are you?"]
enc_inp  = tokenizer(tagged, return_tensors='pt', padding=True, truncation=True, max_length=256)
with torch.no_grad():
    enc_wrapper_pt = model.model.encoder
    enc_out = enc_wrapper_pt(
        input_ids=enc_inp['input_ids'],
        attention_mask=enc_inp['attention_mask']
    ).last_hidden_state

# Multi-token seed: [2, 41854, 5, 149]  (</s> नमस्ते , आप)
decoder_input_ids = torch.tensor([[2, 41854, 5, 149]])

torch.onnx.export(
    dec_wrapper,
    (decoder_input_ids, enc_out, enc_inp['attention_mask']),
    f'{ONNX_DIR}/decoder_model.onnx',
    input_names  = ['decoder_input_ids', 'encoder_hidden_states', 'encoder_attention_mask'],
    output_names = ['logits'],
    dynamic_axes = {
        'decoder_input_ids':      {0: 'batch', 1: 'dec_seq'},
        'encoder_hidden_states':  {0: 'batch', 1: 'enc_seq'},
        'encoder_attention_mask': {0: 'batch', 1: 'enc_seq'},
        'logits':                 {0: 'batch', 1: 'dec_seq'},
    },
    opset_version       = 14,
    do_constant_folding = True,
    dynamo=False,
)
print('decoder_model.onnx re-exported ✓')

# Verify
m = onnx.load(f'{ONNX_DIR}/decoder_model.onnx')
onnx.checker.check_model(m)
print(f'checker passed  nodes={len(m.graph.node)} ✓')

# ── 3. Smoke-test both graphs ─────────────────────────────────────────────────
print('\nRunning onnx.checker ...')
for name in ['encoder_model.onnx', 'decoder_model.onnx']:
    m = onnx.load(f'{ONNX_DIR}/{name}')
    onnx.checker.check_model(m)
    print(f'  {name}  nodes={len(m.graph.node)}  ✓')

# ── 4. Copy tokenizer files ───────────────────────────────────────────────────
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download

print('\nCopying tokenizer files ...')
cache = snapshot_download(MODEL_ID)   # ← remove trust_remote_code, not a valid arg
for f in Path(cache).iterdir():
    if f.suffix in ('.json', '.model', '.SRC', '.TGT') or f.name.startswith('sentencepiece'):
        shutil.copy(f, ONNX_DIR)
        print(f'  copied {f.name}')

print('\nCell 3 complete — ready for quantization.')

Exporting encoder ...


/tmp/ipykernel_13618/22433689.py:38: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/root/.cache/huggingface/modules/transformers_modules/ai4bharat/indictrans2-en-indic-dist-200M/173b94239f7c38886b2747b8d4a5db771a7e1232/modeling_indictrans.py:186: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if max_pos > self.weights.size(0):
/root/.cache/huggingface/modules/transformers_modules/ai4bharat/indictrans2-en-indic-dist-200M/1

  encoder_model.onnx ✓
Exporting decoder ...


/tmp/ipykernel_13618/22433689.py:96: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:86: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if input_shape[-1] > 1 or self.sliding_window is not None:
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:162: TracerWarning: Converting a tensor to a Python boolean might caus

decoder_model.onnx re-exported ✓
checker passed  nodes=4696 ✓

Running onnx.checker ...
  encoder_model.onnx  nodes=2581  ✓
  decoder_model.onnx  nodes=4696  ✓

Copying tokenizer files ...


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

  copied model.SRC
  copied special_tokens_map.json
  copied config.json
  copied tokenizer_config.json
  copied generation_config.json
  copied dict.SRC.json
  copied dict.TGT.json
  copied model.TGT

Cell 3 complete — ready for quantization.


In [10]:
# ── CELL 4 — INT8 dynamic quantization ───────────────────────────────────────
#
# FIX 1: Output filenames are kept identical to input filenames.
#   Original notebook renamed to *_quantized.onnx, which broke
#   ORTModelForSeq2SeqLM.from_pretrained() — it expects the standard names.
#
# FIX 2: optimize_model=False.
#   With optimize_model=True the quantizer fuses MatMul nodes first, then
#   silently skips quantizing them because they are no longer bare MatMuls.
#   Result: the output file looks smaller but is running FP32 at runtime.
#   The correct order is: export → quantize → then use ORT's session-level
#   optimization (graph_optimization_level) at inference time.
#
# FIX 3: MatMulInteger node count check.
#   Verifies that quantization actually produced INT8 weight ops, not FP32.
#
# FIX 4: Required tokenizer files verified after copy.

from onnxruntime.quantization import quantize_dynamic, QuantType
import glob, shutil, os, onnx

assert 'ONNX_DIR'  in dir(), 'Run Cell 0 first.'
assert 'QUANT_DIR' in dir(), 'Run Cell 0 first.'

onnx_files = sorted(glob.glob(f'{ONNX_DIR}/*.onnx'))
print(f'Quantizing {len(onnx_files)} ONNX files ...')

for src_path in onnx_files:
    fname    = os.path.basename(src_path)          # keep the ORIGINAL filename
    dst_path = f'{QUANT_DIR}/{fname}'
    print(f'  {fname} ...', end=' ', flush=True)

    quantize_dynamic(
        src_path,
        dst_path,
        weight_type=QuantType.QInt8,
        # optimize_model=False,   # FIX: do NOT pre-fuse before quantizing
    )

    # Verify quantization produced MatMulInteger ops (not 0, which means FP32 fallback)
    m    = onnx.load(dst_path)
    ops  = [n.op_type for n in m.graph.node]
    n_mmi = ops.count('MatMulInteger')
    if n_mmi == 0:
        raise RuntimeError(
            f'Quantization produced 0 MatMulInteger nodes in {fname}.\n'
            f'This means the model is still FP32. Check onnxruntime version compatibility.'
        )
    print(f'MatMulInteger nodes = {n_mmi}  ✓')

# ── Copy all non-ONNX files (tokenizer, configs) ──────────────────────────────
print('\nCopying tokenizer and config files ...')
for f in os.listdir(ONNX_DIR):
    src_path = f'{ONNX_DIR}/{f}'

    if os.path.isfile(src_path) and not f.endswith('.onnx'):
        shutil.copy2(src_path, f'{QUANT_DIR}/{f}')
        print(f'  copied: {f}')

# ── Verify all required files are present in QUANT_DIR ────────────────────────
required_all = [
    'encoder_model.onnx',
    'decoder_model.onnx',
    'tokenizer_config.json',
    'special_tokens_map.json',
]

print('\nVerifying required files in QUANT_DIR ...')
for name in required_all:
    path = f'{QUANT_DIR}/{name}'
    assert os.path.exists(path), f'MISSING: {path}'
    sz = os.path.getsize(path) / 1e6
    print(f'  {name:<45}  {sz:.1f} MB  ✓')

quant_files = sorted(os.listdir(QUANT_DIR))
total_mb    = sum(os.path.getsize(f'{QUANT_DIR}/{f}') for f in quant_files) / 1e6
print(f'\nQuantization complete. QUANT_DIR total: {total_mb:.0f} MB  (target < 200 MB)')
if total_mb > 200:
    print('  ⚠️  Size exceeds 200 MB target — consider pruning or a smaller checkpoint.')

Quantizing 2 ONNX files ...
  decoder_model.onnx ... 

MatMulInteger nodes = 181  ✓
  encoder_model.onnx ... 

MatMulInteger nodes = 108  ✓

Copying tokenizer and config files ...
  copied: model.SRC
  copied: special_tokens_map.json
  copied: config.json
  copied: tokenizer_config.json
  copied: generation_config.json
  copied: dict.SRC.json
  copied: dict.TGT.json
  copied: model.TGT

Verifying required files in QUANT_DIR ...
  encoder_model.onnx                             74.9 MB  ✓
  decoder_model.onnx                             203.7 MB  ✓
  tokenizer_config.json                          0.0 MB  ✓
  special_tokens_map.json                        0.0 MB  ✓

Quantization complete. QUANT_DIR total: 287 MB  (target < 200 MB)
  ⚠️  Size exceeds 200 MB target — consider pruning or a smaller checkpoint.


In [ ]:
import os
for f in ['model.safetensors', 'pytorch_model.bin', 'README.md', '.gitattributes', 'LICENSE']:
    p = f'{QUANT_DIR}/{f}'
    if os.path.exists(p):
        os.remove(p)
        print(f'removed {f}')

# Confirm real size
import os
total = sum(os.path.getsize(f'{QUANT_DIR}/{f}') for f in os.listdir(QUANT_DIR)) / 1e6
print(f'QUANT_DIR actual size: {total:.0f} MB')

In [12]:
# ── CELL 5 — Validate quantized model + latency benchmark + zip ───────────────

import time, os, shutil, sacrebleu
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

assert 'SRC' in dir(), 'Run Cell 0 first.'
for var in ['baseline_bleu_greedy', 'baseline_chrf_greedy', 'hyps_greedy']:
    assert var in dir(), f'Run Cell 2 first (missing: {var}).'

# ── Load tokenizer ────────────────────────────────────────────────────────────
print('Loading tokenizer ...')

tok_ort = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

# ── Load ONNX sessions ────────────────────────────────────────────────────────
print('Loading ONNX sessions ...')
so = ort.SessionOptions()
so.intra_op_num_threads = 2
so.inter_op_num_threads = 1

enc_sess = ort.InferenceSession(f'{QUANT_DIR}/encoder_model.onnx', sess_options=so)
dec_sess = ort.InferenceSession(f'{QUANT_DIR}/decoder_model.onnx', sess_options=so)
print('Sessions loaded.')

# ── Greedy decode ─────────────────────────────────────────────────────────────
def translate_ort(sentences, max_new_tokens=64):
    tagged  = [f"{SRC_LANG} {TGT_LANG} {s}" for s in sentences]
    enc_inp = tok_ort(tagged, return_tensors='np', padding=True,
                      truncation=True, max_length=256)

    enc_out = enc_sess.run(
        ['last_hidden_state'],
        {'input_ids':      enc_inp['input_ids'],
         'attention_mask': enc_inp['attention_mask']}
    )[0]

    batch     = len(sentences)
    eos_id    = tok_ort.eos_token_id or 2
    dec_ids   = np.full((batch, 1), 2, dtype=np.int64)  # seed
    generated = [[] for _ in range(batch)]
    done      = [False] * batch

    for _ in range(max_new_tokens):
        logits = dec_sess.run(
            ['logits'],
            {'decoder_input_ids':      dec_ids,          # full history
             'encoder_hidden_states':  enc_out,
             'encoder_attention_mask': enc_inp['attention_mask']}
        )[0]   # (batch, dec_seq, vocab)

        next_token = logits[:, -1, :].argmax(axis=-1)   # last position only

        all_done = True
        for b in range(batch):
            if not done[b]:
                tid = int(next_token[b])
                if tid == eos_id:
                    done[b] = True
                else:
                    generated[b].append(tid)
                    all_done = False

        dec_ids = np.concatenate(
            [dec_ids, next_token[:, np.newaxis]], axis=1
        )

        if all_done:
            break

    return tok_ort.batch_decode(generated, skip_special_tokens=True)

# ── Quality metrics ───────────────────────────────────────────────────────────
print('\nRunning quantized translations ...')
hyps_ort  = translate_ort(SRC)

bleu_ort  = sacrebleu.corpus_bleu(hyps_ort, [REF]).score
chrf_ort  = sacrebleu.corpus_chrf(hyps_ort, [REF]).score
bleu_drop = baseline_bleu_greedy - bleu_ort
chrf_drop = baseline_chrf_greedy - chrf_ort

print(f'\n── Quality metrics (ORT INT8 greedy vs PyTorch greedy) ──')
print(f'PyTorch BLEU  : {baseline_bleu_greedy:.2f}')
print(f'ORT INT8 BLEU : {bleu_ort:.2f}   drop = {bleu_drop:+.2f}')
print(f'PyTorch chrF++: {baseline_chrf_greedy:.2f}')
print(f'ORT INT8 chrF+: {chrf_ort:.2f}   drop = {chrf_drop:+.2f}')

# ── Quality gate ──────────────────────────────────────────────────────────────
BLEU_DROP_LIMIT = 3.0
CHRF_DROP_LIMIT = 6.0

assert bleu_drop < BLEU_DROP_LIMIT, f'BLEU drop {bleu_drop:.2f} exceeds limit!'
assert chrf_drop < CHRF_DROP_LIMIT, f'chrF++ drop {chrf_drop:.2f} exceeds limit!'
print('Quality gate passed. ✅')

# ── Per-sentence spot-check ───────────────────────────────────────────────────
print('\n── Per-sentence spot-check ──')
for i, (src, hyp_ort, hyp_base, ref) in enumerate(zip(SRC, hyps_ort, hyps_greedy, REF)):
    s_base = sacrebleu.sentence_bleu(hyp_base, [ref]).score
    s_ort  = sacrebleu.sentence_bleu(hyp_ort,  [ref]).score
    drop   = s_base - s_ort
    flag   = '  ⚠️' if drop > 10 else ''
    print(f'  [{i:02d}] base={s_base:5.1f}  ort={s_ort:5.1f}  drop={drop:+5.1f}{flag}')
    if drop > 10:
        print(f'       EN : {src}')
        print(f'       ORT: {hyp_ort}')
        print(f'       REF: {ref}')

# ── Latency benchmark ─────────────────────────────────────────────────────────
print('\n── Latency benchmark (per-sentence, greedy) ──')
latencies_ms = []
for s in SRC:
    t0 = time.perf_counter()
    translate_ort([s])
    latencies_ms.append((time.perf_counter() - t0) * 1000)

latencies_ms.sort()
p50  = latencies_ms[len(latencies_ms) // 2]
p95  = latencies_ms[int(len(latencies_ms) * 0.95)]
pmax = latencies_ms[-1]
print(f'  P50 : {p50:.0f} ms')
print(f'  P95 : {p95:.0f} ms')
print(f'  Max : {pmax:.0f} ms')
if p50 > 400:
    print('  ⚠️  P50 > 400 ms — may feel slow for a keyboard app.')
else:
    print('  Latency target met (P50 < 400 ms). ✓')

# ── Output spot-check ─────────────────────────────────────────────────────────
print('\n── Output spot-check ──')
for s, h, r in zip(SRC[:3], hyps_ort[:3], REF[:3]):
    print(f'  EN  : {s}')
    print(f'  ORT : {h}')
    print(f'  REF : {r}')
    print()

# ── Clean up bloat before zipping ─────────────────────────────────────────────
for f in ['model.safetensors', 'pytorch_model.bin', 'README.md',
          '.gitattributes', 'LICENSE']:
    p = f'{QUANT_DIR}/{f}'
    if os.path.exists(p):
        os.remove(p)

# ── Zip ───────────────────────────────────────────────────────────────────────
print('Creating zip ...')
shutil.make_archive(ZIP_PATH, 'zip', QUANT_DIR)
zip_mb = os.path.getsize(ZIP_PATH + '.zip') / 1e6
print(f'✅  {ZIP_PATH}.zip  ({zip_mb:.0f} MB)')

Loading tokenizer ...
Loading ONNX sessions ...
Sessions loaded.

Running quantized translations ...

── Quality metrics (ORT INT8 greedy vs PyTorch greedy) ──
PyTorch BLEU  : 37.64
ORT INT8 BLEU : 35.71   drop = +1.92
PyTorch chrF++: 82.52
ORT INT8 chrF+: 77.90   drop = +4.62
Quality gate passed. ✅

── Per-sentence spot-check ──
  [00] base=100.0  ort=100.0  drop= +0.0
  [01] base=  9.7  ort=  9.7  drop= +0.0
  [02] base= 39.8  ort= 39.8  drop= +0.0
  [03] base= 18.0  ort= 18.0  drop= +0.0
  [04] base= 50.8  ort= 30.2  drop=+20.6  ⚠️
       EN : I love you.
       ORT: मैं तुमसे प्यार करती हूँ ।
       REF: मैं तुमसे प्यार करता हूँ।
  [05] base=  9.7  ort=  9.7  drop= +0.0
  [06] base=100.0  ort= 86.7  drop=+13.3  ⚠️
       EN : Can you help me?
       ORT: क्या आप मेरी मदद कर सकते हैं
       REF: क्या आप मेरी मदद कर सकते हैं?
  [07] base= 38.3  ort=  9.4  drop=+28.8  ⚠️
       EN : I am feeling sick today.
       ORT: आज मैं बीमार हूँ ।
       REF: मैं आज बीमार महसूस कर रहा हूँ।
  [0

In [14]:
# ── FINAL COMPARISON: PyTorch vs ONNX INT8 ─────────────────────────────

import time
import torch

model.eval()

# ── PyTorch inference (greedy, SAME as ONNX) ───────────────────────────

def translate_pt(sentences, max_new_tokens=64):
    tagged = [f"{SRC_LANG} {TGT_LANG} {s}" for s in sentences]

    inputs = tokenizer(
        tagged,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=1,
            do_sample=False
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# ── Benchmark function ─────────────────────────────────────────────────

def benchmark(fn, name):
    latencies = []

    for s in SRC:
        t0 = time.perf_counter()
        fn([s])
        latencies.append((time.perf_counter() - t0) * 1000)

    latencies.sort()
    p50 = latencies[len(latencies)//2]
    p95 = latencies[int(len(latencies)*0.95)]
    pmax = latencies[-1]

    print(f"\n{name}")
    print(f"  P50 : {p50:.0f} ms")
    print(f"  P95 : {p95:.0f} ms")
    print(f"  Max : {pmax:.0f} ms")

    return p50, p95

# ── Warmup ────────────────────────────────────────────────────────────

print("Warming up...")
translate_pt([SRC[0]])
translate_ort([SRC[0]])

# ── Run benchmarks ────────────────────────────────────────────────────

print("\nRunning comparison...\n")

pt_p50, pt_p95   = benchmark(translate_pt,  "PyTorch (200M)")
ort_p50, ort_p95 = benchmark(translate_ort, "ONNX INT8")

# ── Speedup ───────────────────────────────────────────────────────────

print("\n── Speedup ──")
print(f"P50 speedup: {pt_p50 / ort_p50:.2f}x")
print(f"P95 speedup: {pt_p95 / ort_p95:.2f}x")

# ── Summary ───────────────────────────────────────────────────────────

print("\n── Summary ──")
print(f"Latency reduced from {pt_p50:.0f} ms → {ort_p50:.0f} ms")
print(f"Improvement: {(1 - ort_p50/pt_p50)*100:.1f}%")

# ── Output comparison ─────────────────────────────────────────────────

print("\n── Output comparison ──")

for s in SRC[:3]:
  pt_out  = translate_pt([s])[0]
  ort_out = translate_ort([s])[0]

  print(f"EN : {s}")
  print(f"PT : {pt_out}")
  print(f"ORT: {ort_out}")
  print()



Warming up...

Running comparison...


PyTorch (200M)
  P50 : 579 ms
  P95 : 1000 ms
  Max : 1000 ms

ONNX INT8
  P50 : 290 ms
  P95 : 736 ms
  Max : 736 ms

── Speedup ──
P50 speedup: 2.00x
P95 speedup: 1.36x

── Summary ──
Latency reduced from 579 ms → 290 ms
Improvement: 49.9%

── Output comparison ──
EN : Hello, how are you?
PT : नमस्ते, आप कैसे हैं?
ORT: नमस्ते, आप कैसे हैं?

EN : I will be late today.
PT : आज मुझे देर हो जाएगी ।
ORT: आज मुझे देर हो जाएगी ।

EN : Please send me the document.
PT : कृपया मुझे दस्तावेज़ भेजें ।
ORT: कृपया मुझे दस्तावेज़ भेजें ।

